In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""# take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    return f"Email sent"


In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_ollama import ChatOllama

class EmailState(AgentState):
    email: str
llm = ChatOllama(model="llama3.2:3b")

agent = create_agent(
    model=llm,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [11]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}
llm = ChatOllama(model="llama3.2:3b")
response = agent.invoke(
    {
    "messages": [HumanMessage(content="Veuillez lire mon e-mail et envoyer une réponse immédiatement. Envoyez la réponse maintenant dans le même fil de discussion.")],
    "email": "Bonjour Sara, je vais être en retard pour notre réunion de demain. Pouvons-nous la reprogrammer ? Cordialement, Sofia"
    },
    config=config
)
print(response)


{'messages': [HumanMessage(content='Veuillez lire mon e-mail et envoyer une réponse immédiatement. Envoyez la réponse maintenant dans le même fil de discussion.', additional_kwargs={}, response_metadata={}, id='f4737d77-bc98-4060-aea1-460024eaa06f'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-15T22:35:31.8596998Z', 'done': True, 'done_reason': 'stop', 'total_duration': 155855365200, 'load_duration': 118400882200, 'prompt_eval_count': 218, 'prompt_eval_duration': 17409329000, 'eval_count': 40, 'eval_duration': 17199016000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019ecd6a-996e-7d72-a981-47a28236ff99-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '6b3d18e2-fbd0-4266-a21f-215e7c148b8d', 'type': 'tool_call'}, {'name': 'send_email', 'args': {'body': 'Répondre immédiatement à mon e-mail'}, 'id': '162f5f38-fc07-4a4a-b147-61ceebe1f31e', 'type': 'tool_call'}], invalid_tool

In [5]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Répondre immédiatement à mon e-mail'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Répondre immédiatement à mon e-mail'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='6334598ae39091f83ce611660a01b52b')]


In [8]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Répondre immédiatement à mon e-mail


In [9]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Le même thread ID pour reprendre la conversation
)
print(response['messages'][-1].content)


Voilà, j'ai envolé un email. Avez-vous besoin d'aide supplémentaire?


In [10]:
response = agent.invoke(
    Command(
        resume={
            "decisions": [
            {
                "type": "reject",
                # Une explication sur les raisons du rejet
                "message": " J’annule notre rendez-vous."
            }
            ]
        }
    ),
    config=config # Le même thread ID pour reprendre la conversation
)
print(response)

{'messages': [HumanMessage(content='Veuillez lire mon e-mail et envoyer une réponse immédiatement. Envoyez la réponse maintenant dans le même fil de discussion.', additional_kwargs={}, response_metadata={}, id='f4737d77-bc98-4060-aea1-460024eaa06f'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-15T22:35:31.8596998Z', 'done': True, 'done_reason': 'stop', 'total_duration': 155855365200, 'load_duration': 118400882200, 'prompt_eval_count': 218, 'prompt_eval_duration': 17409329000, 'eval_count': 40, 'eval_duration': 17199016000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019ecd6a-996e-7d72-a981-47a28236ff99-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '6b3d18e2-fbd0-4266-a21f-215e7c148b8d', 'type': 'tool_call'}, {'name': 'send_email', 'args': {'body': 'Répondre immédiatement à mon e-mail'}, 'id': '162f5f38-fc07-4a4a-b147-61ceebe1f31e', 'type': 'tool_call'}], invalid_tool

In [12]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        # Le nom du Tool.
                        "name": "send_email",
                        # Les arguments à passer au tool.
                        "args": {"body": "Je suis désolée mais je dois annuler notre rendez-vous je ne serais pas libre. Sara"},
                    }
                }
            ]
        }
    ),
    config=config # Le même thread ID pour reprendre la conversation
    )
print(response['messages'][-1].content)

{"name": "send_email", "parameters": {"body": "Répondre imm\u00e9diatement \u00e0 mon e-mail, j'ai envol\u00e9 un email. Avez-vous besoin d"aide suppl\u00e9mentaire?"}}
